# Turn Detection and Endpointing

Simulate calls, look at the pauses, and replay them through three endpointing policies. Predict each result first.

In [ ]:
import sys
sys.path.insert(0, '../code')
import numpy as np
import main

calls = main.simulate_calls()
pauses = main.within_turn_pauses(calls)
for timer in (0.3, 0.5, 0.8, 1.2):
    print(f'{timer:.1f} s timer is shorter than {100 * np.mean(pauses > timer):4.1f}% of pauses inside turns')

## The toy end-of-utterance model

Type your own transcripts. Which cue does each one trigger?

In [ ]:
for text in ('số điện thoại của em là 0903', 'số điện thoại của em là 0903 123 456',
             'em muốn hỏi về', 'cho em gặp nhân viên', 'dạ vâng'):
    print(f'{main.eou_probability(text):.2f}', main.eou_features(text), text)

## The trade-off

Sweep the fixed timer, then compare the adaptive policies at similar waiting times.

In [ ]:
for delay in (0.3, 0.5, 0.8, 1.2, 2.0):
    r = main.run_policy(main.FixedEndpointing(delay), calls)
    print(f"fixed {delay:.1f} s: cut off {100 * r['cutoff_rate']:4.1f}%  wait {r['mean_wait_s']:.2f} s")
for name, policy in (('dynamic', main.DynamicEndpointing(0.5, 3.0)), ('semantic', main.SemanticEndpointing(0.5, 3.0))):
    r = main.run_policy(policy, calls)
    print(f"{name}: cut off {100 * r['cutoff_rate']:4.1f}%  wait {r['mean_wait_s']:.2f} s  p95 {r['p95_wait_s']:.2f} s")

## One slow caller

Give a single caller a slow tempo and watch the dynamic timer adapt.

In [ ]:
slow = [main.simulate_call(n_turns=30, seed=3, tempo=2.0)]
for name, policy in (('fixed 0.5', main.FixedEndpointing(0.5)), ('dynamic', main.DynamicEndpointing(0.5, 3.0)),
                     ('semantic', main.SemanticEndpointing(0.5, 3.0))):
    r = main.run_policy(policy, slow)
    print(f"{name:>9}: cut off {100 * r['cutoff_rate']:4.1f}%  wait {r['mean_wait_s']:.2f} s")